In [0]:
catalog = "workspace"
bronze_schema_name = "bronze"
silver_schema_name = "silver"
gold_schema_name = "gold"

bronze_schema = f"{catalog}.{bronze_schema_name}"
silver_schema = f"{catalog}.{silver_schema_name}"
gold_schema = f"{catalog}.{gold_schema_name}"

In [0]:
df_silver_info = spark.table(f"{catalog}.silver.info")
df_silver_financials = spark.table(f"{catalog}.silver.financials")
df_silver_metrics = spark.table(f"{catalog}.silver.metrics")
df_silver_cotacao_dolar = spark.table(f"{catalog}.silver.cotacao_dolar")
df_silver_generos = spark.table(f"{catalog}.silver.generos")
df_silver_pessoas = spark.table(f"{catalog}.silver.pessoas_empresas")
df_silver_reviews = spark.table(f"{catalog}.silver.reviews")
                                      
display(spark.table(f"{silver_schema}.info").limit(5))

Consolidação de Fatos e Métricas: Centralização de todas as métricas numéricas do negócio (financeiras, popularidade e contagem de votos) em uma única Tabela de Fatos. Realização de joins com a tabela de câmbio usando a data de lançamento e geração da Surrogate Key (sk_movie_id) para otimizar o relacionamento relacional na camada Gold. Aplicação de tipagem estrita (casting para decimal e int) para garantir exatidão nos cálculos analíticos

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
df_base_filmes=(
    df_silver_info
    .join(df_silver_financials, on='id_filme', how='inner')
    .join(df_silver_metrics, on='id_filme', how='inner') 
)
df_com_cotacao = (
    df_base_filmes
    .join(
        df_silver_cotacao_dolar,
        df_base_filmes["data_lancamento"] == df_silver_cotacao_dolar["data"],
        "left"
    )
)
janela_sk = Window.partitionBy(F.lit(1)).orderBy("id_filme")

df_fact_movies_performance = (
    df_com_cotacao
    .withColumn("sk_movie_id", F.row_number().over(janela_sk).cast("bigint"))
    .select(
        "sk_movie_id",
        F.col("id_filme"),
        F.col("orcamento_usd").cast("decimal(18,2)"),
        F.col("receita_usd").cast("decimal(18,2)"),
        F.col("lucro_usd").cast("decimal(18,2)"),
        F.col("orcamento_brl").cast("decimal(18,2)"),
        F.col("receita_brl").cast("decimal(18,2)"),
        F.col("lucro_brl").cast("decimal(18,2)"),
        F.col("margem_lucro_percentual").cast("decimal(18,2)"),
        F.col("popularidade").cast("double"),
        F.col("nota_media_tmdb").cast("double"),
        F.col("qtd_votos_tmdb").cast("int"),
        F.col("nota_media_imdb").cast("double"),
        F.col("qtd_votos_imdb").cast("int")
    )
    .dropDuplicates(["id_filme"])
)
display(df_fact_movies_performance.limit(20))
df_fact_movies_performance.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{catalog}.gold.fact_movies_performance")

Dimensão Central de Filmes: Isolamento dos atributos puramente descritivos de cada obra (título, idioma, status, sinopse, duração). O uso da mesma lógica de Surrogate Key aplicada na tabela de fatos garante o alinhamento perfeito do modelo Star Schema, aliviando a tabela de fatos do armazenamento de strings pesadas.

In [0]:
janela_sk = Window.partitionBy(F.lit(1)).orderBy("id_filme")
df_dim_movies= (
    df_silver_info
    .withColumn("sk_movie_id", F.row_number().over(janela_sk).cast("bigint"))
    .select(
        "sk_movie_id",
        F.col("id_filme").cast("string"),
        F.col("titulo"),
        F.col("titulo_original"),
        F.col("data_lancamento"),
        F.col("duracao_minutos"),
        F.col("idioma_original"),
        F.col("status_filme"),
        F.col("sinopse")
    )
    .dropDuplicates(["id_filme"])
)
display(df_dim_movies.limit(20))
df_dim_movies.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{catalog}.gold.dim_movies"
)

Dimensão de Gêneros: Extração e deduplicação de todas as categorias de gênero únicas validadas na camada Silver. Geração de uma Surrogate Key exclusiva (sk_genre_id) para suportar relacionamentos múltiplos.

In [0]:
df_generos = (
    df_silver_generos
    .select(F.col("genero").alias("nome_genero"))
    .dropDuplicates(["nome_genero"])
)
janela_sk = Window.partitionBy(F.lit(1)).orderBy("nome_genero")
df_dim_generos = (
    df_generos
    .withColumn("sk_genre_id", F.row_number().over(janela_sk).cast("bigint"))
    .select(
        "sk_genre_id",
        "nome_genero"
    )
)
display(df_dim_generos.limit(20))
df_dim_generos.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{catalog}.gold.dim_generos"
)

Dimensão de Pessoas Físicas: Isolamento dos profissionais envolvidos nos filmes (filtrado estritamente para 'Diretor', 'Roteirista' e 'Ator'). A regra de deduplicação utiliza a chave composta de "nome" e "cargo", garantindo que a mesma pessoa física possa ser rastreada corretamente caso atue em múltiplos papéis distintos.

In [0]:
df_pessoas = (
    df_silver_pessoas
    .select(
        F.col("nome").alias("nome_pessoa"),
        F.col("tipo_entidade").alias("tipo_pessoa")
    )
    .filter(
        F.col("nome_pessoa").isNotNull()&
        (F.col("nome_pessoa") != "")&
        F.col("tipo_pessoa").isin(["Diretor", "Roteirista", "Ator"])
    )
    .dropDuplicates(["nome_pessoa", "tipo_pessoa"])
)
janela_sk = Window.partitionBy(F.lit(1)).orderBy("nome_pessoa")
df_dim_people = (
    df_pessoas
    .withColumn("sk_person_id",F.row_number().over(janela_sk).cast("bigint"))
    .select(
        "sk_person_id",
        "nome_pessoa",
        "tipo_pessoa"
    )
)
display(df_dim_people)
df_dim_people.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{catalog}.gold.dim_people"

)

Dimensão de Organizações: Construção da dimensão de empresas produtoras. Aplicação de uma rigorosa camada adicional de qualidade de dados com expressões regulares (Regex) para expurgar caracteres residuais (como # e &) e anomalias de nomenclatura antes da geração da Surrogate Key (sk_company_id).

In [0]:
df_produtoras = (df_silver_pessoas
    .filter(F.col("tipo_entidade") == "Produtora")
    .select(
    F.col("nome").alias("nome_produtora"))
    .filter(F.col("nome_produtora").isNotNull() & (F.col("nome_produtora") != ""))
    .withColumn("nome_produtora", F.regexp_replace(F.col("nome_produtora"), r"^[#&]\s*", ""))
    .withColumn("nome_produtora", F.initcap(F.trim(F.col("nome_produtora"))))
    .filter(~F.col("nome_produtora").rlike(r'^\d+(\.\d+)?$')) 
    .filter(~F.col("nome_produtora").rlike(r'\.\s+[A-Z]'))
    .filter(~F.col("nome_produtora").rlike(r'//|^[\d/]+$'))
    .filter(F.length(F.col("nome_produtora")) > 0)
    .dropDuplicates(["nome_produtora"])
)
janela_sk = Window.partitionBy(F.lit(1)).orderBy("nome_produtora")
df_dim_producers = (
    df_produtoras
    .withColumn("sk_company_id",F.row_number().over(janela_sk).cast("bigint"))
    .select(
        "sk_company_id",
        "nome_produtora"
    )
)
display(df_dim_producers)
df_dim_producers.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold.dim_producers")



Agregação de Avaliações (Reviews): Resumo analítico a nível de filme de todos os comentários individuais dos usuários. Cálculo do volume total de avaliações e da média final das notas atribuídas (nota_media_usuarios), criando um perfil de aceitação do público que se conecta à dimensão principal de filmes via Foreign Key.

In [0]:
df_reviews = (
    df_silver_reviews
    .groupby("id_filme")
    .agg(
        F.count("*").cast("int").alias("qnt_avaliacoes_usuarios"),
        F.round(F.avg("nota_usuario"),2).cast("double").alias("nota_media_usuarios"))
    )
df_dim_movies = spark.table(f"{catalog}.gold.dim_movies").select("sk_movie_id", "id_filme")
df_dim_reviews_fk = (
    df_reviews
    .join(df_dim_movies, on="id_filme",how="inner")
    .select("sk_movie_id", "qnt_avaliacoes_usuarios", "nota_media_usuarios")
)
janela_sk_review = Window.partitionBy(F.lit(1)).orderBy("sk_movie_id")
df_dim_reviews = (
    df_dim_reviews_fk
    .withColumn("sk_review_id",F.row_number().over(janela_sk_review).cast("bigint"))
    .select(
        "sk_review_id",
        "sk_movie_id",
        "qnt_avaliacoes_usuarios",
        "nota_media_usuarios"
    )
)
display(df_dim_reviews.limit(20))
df_dim_reviews.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{catalog}.gold.dim_reviews")





Resolução de Cardinalidade N:N (Gêneros): Implementação de uma tabela-ponte (Bridge Table) para suportar a relação de Muitos-para-Muitos, dado que um filme pode ter múltiplos gêneros.

In [0]:
df_silver_generos = spark.read.table(f"{catalog}.silver.generos").select("id_filme",F.col("genero").alias("nome_genero"))
df_bridge_movie_genre = (
    df_silver_generos
    .join(df_dim_movies, on="id_filme", how="inner")
    .join(df_dim_generos, on = "nome_genero", how="inner")
    .select("sk_movie_id", "sk_genre_id")
    .dropDuplicates()
)
display(df_bridge_movie_genre)
df_bridge_movie_genre.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{catalog}.gold.bridge_movie_genre")

Resolução de Cardinalidade N:N (Equipe): Criação da tabela-ponte associando filmes aos seus respectivos atores, diretores e roteiristas. O join utiliza o cargo e o nome para capturar a Surrogate Key correta da pessoa na dimensão.

In [0]:
df_silver_pessoas = (spark.table(f"{catalog}.silver.pessoas_empresas")
    .filter(F.col("tipo_entidade")!= "Produtora")
    .select("id_filme",F.col("nome").alias("nome_pessoa"),
    F.col("tipo_entidade").alias("tipo_pessoa")))
df_brigde_movie_person = (
    df_silver_pessoas
    .join(df_dim_movies, on="id_filme", how="inner")
    .join(df_dim_people, on = ["nome_pessoa","tipo_pessoa"], how="inner")
    .select("sk_movie_id", "sk_person_id")
    .dropDuplicates()
    
)
display(df_brigde_movie_person)
df_brigde_movie_person.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{catalog}.gold.bridge_movie_person")

Resolução de Cardinalidade N:N (Empresas): Construção da ponte entre os filmes e as produtoras. Replica-se o tratamento de strings da dimensão (limpeza de caracteres) no momento do join com os dados da Silver, garantindo correspondência exata e prevenindo a perda de registros no modelo Gold.

In [0]:
df_silver_pessoas = (spark.table(f"{catalog}.silver.pessoas_empresas")
    .filter(F.col("tipo_entidade")== "Produtora")
    .select("id_filme",F.col("nome").alias("nome_produtora"))
    .withColumn("nome_produtora", F.regexp_replace(F.col("nome_produtora"), r"^[#&]\s*", ""))
    .withColumn("nome_produtora", F.initcap(F.trim(F.col("nome_produtora")))))
df_bridge_movie_company = (df_silver_pessoas
    .join(df_dim_movies, on="id_filme", how="inner")
    .join(df_dim_producers, on = "nome_produtora", how="inner")
    .select("sk_movie_id", "sk_company_id")
    .dropDuplicates()
)
display(df_bridge_movie_company)
df_bridge_movie_company.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{catalog}.gold.bridge_movie_company")


Agregação global da métrica financeira da receita diretamente da Tabela de Fact, totalizando o valor de todos os filmes da base de dados.

In [0]:
df_receita = spark.table(f"{catalog}.gold.fact_movies_performance")
df_receita.agg(F.sum("receita_brl")).alias("receita_total_em_real").display()

Cruzamento entre a Dimensão de Filmes, para obter o título, e a Tabela de Fact, para acessar à métrica numérica, ordenando os resultados de forma decrescente para revelar os mais populares.

In [0]:
df_populares = spark.table(f"{catalog}.gold.fact_movies_performance")
df_titulo = spark.table(f"{catalog}.gold.dim_movies")
(df_populares
 .join(df_titulo, on="sk_movie_id", how="inner")
 .select("titulo","popularidade")
 .orderBy(F.desc("popularidade"))
 .limit(5)
 .display()

    
)

Utilização da Tabela-Ponte de géneros para contar a cardinalidade correta de filmes por categoria, ordenando do maior para o menor volume e evitando duplicações provocadas por relações de muitos-para-muitos.

In [0]:
df_brig_movie_genre = spark.table(f"{catalog}.gold.bridge_movie_genre")
df_dim_generos = spark.table(f"{catalog}.gold.dim_generos")
(df_bridge_movie_genre.join(df_dim_generos, on="sk_genre_id", how="inner")
.groupBy("nome_genero")
.agg(F.count("sk_movie_id").alias("qtd_filmes"))
.orderBy(F.col("qtd_filmes").desc())
    .display()
    )

Aplicação de uma Window Function com a cláusula estrutural RANK() sobre a Tabela de Fact para classificar os filmes mais rentáveis, efetuando o cruzamento com a dimensão de filmes para visualização do título associado a cada posição no ranking.

In [0]:
ranking = Window.orderBy(F.col("receita_brl").desc())
df_titulo = spark.table(f"{catalog}.gold.dim_movies")
(df_receita
 .join(df_titulo, on="sk_movie_id", how="inner")
 .select("titulo",
         "receita_brl",
         "receita_usd",F.rank().over(ranking).alias("posicao_rank"))
 .orderBy("posicao_rank").limit(10)
 .display()

)

Cálculo dinâmico do limite temporal baseado na data máxima de lançamento existente na base, ignorando datas futuras ou não lançadas. O cruzamento com a Tabela-Ponte e a aplicação do filtro restrito ao cargo "Ator" garantem a contagem correta e isolada de atuações.

In [0]:
df_dim_filmes = spark.table(f"{catalog}.gold.dim_movies")
data_limite_row = df_dim_filmes.filter(F.col("data_lancamento") <= F.current_date()) \
                           .select(F.max("data_lancamento")).collect()[0]
data_limite = data_limite_row[0]

data_corte_2_anos = data_limite - F.expr("INTERVAL 2 YEARS")
data_corte_5_anos = data_limite - F.expr("INTERVAL 5 YEARS")

In [0]:
df_filmes = df_dim_filmes.filter(F.col("data_lancamento").between(data_corte_2_anos, data_limite))
(df_filmes.join(spark.table(f"{catalog}.gold.bridge_movie_person"), on="sk_movie_id", how="inner")
.join(spark.table(f"{catalog}.gold.dim_people"), on="sk_person_id", how="inner")
.filter(F.col("tipo_pessoa") == "Ator")
.groupBy("nome_pessoa")
.agg(F.count("sk_movie_id").alias("qtd_participacoes"))
.orderBy(F.col("qtd_participacoes").desc())
.limit(1)
.display()
)


Cruzamento analítico entre a Tabela de Fact, a Dimensão de Produtoras e a Tabela-Ponte correspondente, aplicando o filtro temporal dinâmico retroativo de 5 anos. A agregação soma o lucro global na tabela de desempenho e identifica a entidade corporativa com o maior retorno financeiro.

In [0]:
df_lucro_filme = df_dim_filmes.filter(F.col("data_lancamento").between(data_corte_5_anos, data_limite))
(df_lucro_filme.join(spark.table(f"{catalog}.gold.bridge_movie_company"), on="sk_movie_id", how="inner") 
.join(spark.table(f"{catalog}.gold.fact_movies_performance"), on="sk_movie_id", how="inner")
.join(spark.table(f"{catalog}.gold.dim_producers"),on = "sk_company_id", how="inner")
.groupBy("nome_produtora") 
.agg(F.sum("lucro_usd").alias("lucro_total_usd"),F.sum("lucro_brl").alias("lucro_total_brl")) 
.orderBy(F.col("lucro_total_usd").desc()) 
.limit(1) 
.display())

